In [0]:
# Cell 1: Import necessary libraries
from pyspark.sql.functions import col


In [0]:
# Cell 2: Create sample data for the Bronze layer
# Create the catalog and schema if they don't exist
spark.sql("CREATE CATALOG IF NOT EXISTS ops")
spark.sql("CREATE SCHEMA IF NOT EXISTS ops.bronze")

data = [
    (1, "John Doe", "2025-09-09", 100.0),
    (2, "Jane Smith", "2025-09-08", 150.0),
    (3, "Sam Brown", "2025-09-07", None)
]

columns = ["id", "name", "date", "amount"]
bronze_df = spark.createDataFrame(data, columns)
bronze_df.write.format("delta").mode("overwrite").saveAsTable("ops.bronze.customer_transactions")


In [0]:
# Cell 3: Read data from the Bronze layer and perform data cleaning for the Silver layer
# Create the silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS ops.silver")

bronze_df = spark.table("ops.bronze.customer_transactions")
silver_df = bronze_df.filter(col("amount").isNotNull())
silver_df.write.format("delta").mode("overwrite").saveAsTable("ops.silver.customer_transactions_cleaned")



In [0]:
# Cell 4: Read data from the Silver layer and perform aggregation for the Gold layer
# Create the gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS ops.gold")

silver_df = spark.table("ops.silver.customer_transactions_cleaned")
gold_df = silver_df.groupBy("name").agg({"amount": "sum", "amount": "avg"}).withColumnRenamed("sum(amount)", "total_spent").withColumnRenamed("avg(amount)", "average_spent")
gold_df.write.format("delta").mode("overwrite").saveAsTable("ops.gold.customer_spending")

In [0]:
# Cell 5: Display the Gold layer data
gold_df = spark.table("ops.gold.customer_spending")
display(gold_df)